# 從官方棋譜重新產生中間檔（分段說明）：競賽二 Rank（v13n）

這本筆記本把管線從官方棋譜一路列到最後一步，也就是**沒有主辦方提供的中間檔時，怎麼用官方資料＋我們的權重＋外部模型重新產生它們**。**每一段都標了硬體需求、原作者實測耗時，以及 Colab 免費版能不能跑完。預設所有重段落都不執行**（`RUN_* = False`）；請依需要打開。

> 先講結論：整條重跑在免費版 Colab 上**做不完**。最上游的整條重跑 GPU 約 71 小時以上、另加微調網路輸入編碼在學校 CPU 約 70 小時（分項相加，沒有整條量過）、認人網路 GPU 小時數十、HumanSL 掃描約 100 萬盤。可以在免費版完成的是：外部權重下載、切分與（小規模）特徵、測試端的抽取、最後一步組合（`reproduce_final.ipynb`，需要中間檔）。
> 官方資料（主辦方提供）不附；自訓權重在 GitHub Release；外部模型（KataGo、HumanSL、SE）不散布，只給官方下載網址。

| 段 | 內容 | 硬體 | 原作者實測耗時（平台） | Colab 免費版（T4、單次 ≤12 小時、約 12 GB RAM） |
|---|---|---|---|---|
| 0 | 安裝、外部權重（約 1.1 GB） | CPU | 約 15 分鐘 | 可 |
| 1 | 切分＋L0 手工特徵快取（100 萬盤） | CPU | 數十分鐘（14 核）；免費版只有 2 核 | **可能跑不完**（核心少、RAM 吃緊；可縮小規模試） |
| 2 | KataGo HumanSL 掃描（訓練端約 100 萬盤，1 visit；後半盤 W120） | GPU（KataGo CUDA） | 約 2.35 盤／秒（Colab L4）：第 1 層 43.8 units；第 2 層約 80 萬盤在學校 RTX8000；後半盤 E35 255,464 盤 28.9 小時 | **不可**（約 118 小時）；測試端 800 題可 |
| 3 | Strength Estimator 抽取 | GPU | 訓練端 685,231 盤 12.9 小時（RTX8000）；測試端約 1 小時 | 訓練端**不可**；測試端可 |
| 4 | KataGo 主幹向量（前半／後半／24 局面 token） | GPU | 約 11 小時（兩 GPU 並行） | **不可** |
| 5 | S 認人網路 finh_full（用來看對手是誰） | GPU | Colab L4 20.8 小時 | **不可** |
| 6 | ft4 讀法網路（HumanSL 解凍最後 4 區塊） | GPU，RAM 大 | 約 11 小時（RTX8000）；K 線編碼約 10 小時量級；中間資料 78 GB | **不可** |
| 7 | 四條腿（M03T2／M04T／A1T2／A1P）＋融合＋對手票＋第二階段 | CPU＋GPU | 小時級 | 部分可（CPU 段） |
| 8 | 最後一步 | CPU | 約 1 分鐘 | 可 → `reproduce_final.ipynb` |

各段的輸入輸出在 `PIPELINE.md` 的步驟表；GPU 結果不會位元重現（`index_add_` 順序不固定、fp16、各平台 torch 版本不同；KataGo 掃描為 1 visit＋隨機對稱，重掃逐盤有雜訊）。CPU 的最後一步則位元重現。
**所以自己重算出來的中間檔，位元組不會和主辦方拿到的那份相同**；要位元組重現最終提交檔，需要主辦方提供的那一份中間檔（見 README）。

In [ ]:
# 設定。標示「待補」的值要等 GitHub repo 與 Release 建好後填入。
import os, sys, subprocess
from pathlib import Path

REPO_URL    = os.environ.get("REPO_URL", "待補")        # 例：https://github.com/<帳號>/aicup2026-go-rank.git
RELEASE_URL = os.environ.get("RELEASE_URL", "待補")     # 我們自訓權重的下載網址，例：https://github.com/<帳號>/aicup2026-go-rank/releases/download/<tag>
REPO_DIR    = Path(os.environ.get("REPO_DIR", "/content/aicup2026-go-rank"))
# （環境變數只是方便本機測試；在 Colab 直接改上面的字串即可。）

def run(cmd, cwd=None, env=None, check=True):
    """執行指令並印出輸出（stdout＋stderr）。"""
    e = dict(os.environ, **(env or {}))
    r = subprocess.run(cmd, cwd=str(cwd or REPO_DIR), env=e, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout[-6000:] if len(r.stdout) > 6000 else r.stdout)
    if check and r.returncode:
        raise RuntimeError(f"指令失敗（exit {r.returncode}）：{cmd}")
    return r


In [ ]:
# 1) 取得程式碼
if not (REPO_DIR / "colab").exists():
    assert REPO_URL != "待補", "請先把上一格的 REPO_URL 填成實際網址"
    run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], cwd="/")
print("repo:", REPO_DIR, "| commit:", run(["git", "rev-parse", "--short", "HEAD"], check=False).stdout.strip())


In [ ]:
# 2) 安裝（CPU 套件鎖版本；GPU 段落另外需要 torch，Colab 內建的即可，原作者版本見 requirements-gpu.txt）
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
try:
    import torch; print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
except ImportError:
    print("沒有 torch（只做 CPU 段落可忽略）")
run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=False)

In [ ]:
# 3) 外部預訓練權重（來源網址、版本、授權見 README「外部資源」；下載後驗 sha256；不散布，只從官方網址取得）
sys.path.insert(0, str(REPO_DIR / "colab"))
import colab_utils as U
RUN_EXTERNAL = False        # True：下載約 1.1 GB（KataGo 執行檔與兩個網路、HumanSL .ckpt、SE go.tar.gz，並解出 se_go.pt）
if RUN_EXTERNAL:
    U.fetch_external(REPO_DIR)
# HumanSL .ckpt 在 torch 2.12 載入前，需用 KataGo 原始碼的 python/katago/train/migrate_defaultdict_to_dict.py 轉換
# （v1.15.0 的 checkpoint 內含 defaultdict）；抽主幹向量時固定用 profile rank_1d、日本規則。細節見各腳本頂端說明。

def show_doc(script, n=1800):
    """印出腳本頂端的說明（參數與輸入輸出以它為準）。"""
    s = (REPO_DIR / script).read_text(encoding="utf-8")
    i = s.find('"""'); j = s.find('"""', i + 3)
    print(f"--- {script} ---\n" + (s[i + 3:j] if i >= 0 and j > i else s[:n])[:n])

## 官方資料放置

主辦方資料不附在 repo。請依 README「官方資料放置」放到 `<repo>/data/`：

```
data/training/                 train_<rank>.csv …（1,000,000 盤）
data/contest_1104/             原測試集（Rank、Player 舊題）
```
Colab 可掛載 Google Drive 後複製或建符號連結；**資料不要放進 GitHub**。

In [ ]:
# 資料就位檢查（只檢查資料夾是否存在）
for d in ("data/training", "data/contest_1104"):
    print(f"{d:22s}", "有" if (REPO_DIR / d).exists() else "沒有")
# from google.colab import drive; drive.mount("/content/drive")      # 若資料在 Drive：掛載後 ln -s 到 data/

## 段 1：資料處理（CPU）

切分一律以 player_id 分組（同一玩家不同時出現在訓練與評估）；L0 手工特徵快取是第一階段的基礎。
輸出：`data/splits/`、`data/cache/l0/`、`data/cache/games_meta.parquet`。

In [ ]:
RUN_DATA = False
ENV = {"PYTHONPATH": "src:scripts:."}
if RUN_DATA:
    run([sys.executable, "scripts/make_split.py"], env=ENV)
    run([sys.executable, "scripts/build_l0_cache.py", "--jobs", "2"], env=ENV)     # 免費版只有 2 核；全量 100 萬盤很可能跑不完
else:
    show_doc("scripts/make_split.py"); show_doc("scripts/build_l0_cache.py")

## 段 2：KataGo HumanSL 掃描（GPU）

固定設定在 `scripts/katago/scan_pins.json` 與 `analysis.cfg`（1 visit、11 個段位 profile、日本規則、前 60 個己方手）。`setup_katago.sh` 在 Colab 安裝 KataGo（CUDA 12.8）與兩個網路（驗證版本）。
掃描由 `scan_runner.py` 以分片（shard）執行、可續跑；訓練端全量約 100 萬盤，免費版做不完。測試端由 `predict_hsl.py` 準備分片（Rank 800 題）。

In [ ]:
RUN_KATAGO_SETUP = False       # 安裝 KataGo 與網路到 /content/katago（需 GPU 執行階段）
RUN_HSL_TEST_PREPARE = False   # 以測試檔準備測試端掃描分片到 data/test_scan（需官方資料）
if RUN_KATAGO_SETUP:
    run(["bash", "scripts/katago/setup_katago.sh"])
if RUN_HSL_TEST_PREPARE:
    run([sys.executable, "scripts/predict_hsl.py", "prepare", "--data-dir", "data/contest_1104", "--root", "data/test_scan"], env={"PYTHONPATH": "src:scripts:.", "GOGAME_COLOR_SEMANTICS": "actual"})
    # 之後逐片執行（可續跑；示意，--shards 依 data/test_scan/shards.parquet 的 id）：
    # run([sys.executable, "scripts/katago/scan_runner.py", "--shards", "S9001", "--sgf-dir", "data/test_scan/sgf", "--shards-table", "data/test_scan/shards.parquet",
    #      "--katago-dir", "/content/katago", "--out", "data/test_scan/out", "--work", "data/test_scan/work", "--platform", "colab"])
    # 入帳與組特徵表：predict_hsl.py ingest DIR → predict_hsl.py features
show_doc("scripts/predict_hsl.py", 900); show_doc("scripts/katago/scan_runner.py", 900)

## 段 3：Strength Estimator 抽取（GPU）

每盤取 32 個己方局面，`score` 頭平均與標準差（fp16、rotation 0）。需要 `data/external/se/extracted/se_go.pt`（見上方外部權重；sha256 `95fc2951…`）。SE 權重由中研院公開頁面下載，我們不再散布。
訓練端（685,231 盤，`scripts/experiments/e21/extract.py`）約 12.9 GPU 小時，免費版做不完；測試端用 `scripts/predict_se.py extract`。

In [ ]:
RUN_SE_SMOKE = False     # 對 Rank 測試檔抽 SE 的小型試跑（--limit 200）；需 GPU、se_go.pt 與官方資料
if RUN_SE_SMOKE:
    run([sys.executable, "scripts/predict_se.py", "extract", "--data-dir", "data/contest_1104", "--out-dir", ".scratch/se_smoke",
         "--model", "data/external/se/extracted/se_go.pt", "--device", "cuda", "--half", "--limit", "200", "--color-semantics", "actual"], env={"PYTHONPATH": "src:scripts:."})
show_doc("scripts/predict_se.py", 1500); show_doc("scripts/experiments/e21/extract.py", 900)

## 段 4–6：主幹向量、S 認人網路、ft4（GPU，免費版做不完）

| 內容 | 腳本（說明在各檔頂端） |
|---|---|
| 主幹向量 | `scripts/oneoff/{e55_prep,e55_trunk,e55_pca,e63_prep,e63_trunk,e63_pca}.py`、`e79_{items,gpu,prep_local,tok}.py` |
| S 認人網路 finh_full | `scripts/experiments/e89/{data_prep,train_s,fastboard,driver}.py`、`e90/{make_data_h,train_e90h,driver_e90h,embed}.py`、`e96/*`（需 numba 0.64.0）。權重 `finh_full` 在 Release，**只做推論不用重訓** |
| ft4 讀法網路 | `scripts/experiments/e98/*`、`e103/{e103_data,e103_train,e103_export,e103_final_data}.py`、`e89/encode_k.py`。權重 `final_s0`（交件版）與 `ft4_s0`（驗證版）在 Release；**由 HumanSL 檢查點微調，使用須遵守 KataGo Neural Network License** |

我們沒有在這本筆記本內提供未驗證的一鍵指令：這幾段的參數很多，請依各腳本頂端的說明與 `PIPELINE.md` 的步驟表執行。下面的格子只印出說明。

In [ ]:
for s in ("scripts/experiments/e89/train_s.py", "scripts/experiments/e90/train_e90h.py", "scripts/experiments/e98/e98_train.py", "scripts/experiments/e103/e103_train.py"):
    show_doc(s, 1200)

## 段 7：四條腿、融合、對手票、第二階段

- 四條腿：`scripts/experiments/e10/pipeline.py`（M03T2 LightGBM／M04T CORAL）、`scripts/oneoff/{e55_a1,e63_a1,e79_a1p}.py`（A1T2／A1P）。**四條腿的模型檔沒有保存**（只剩測試端輸出），要重算就得從訓練端特徵重訓。
- 融合 `e66_fuse.py`、`e79_fuse.py` → 對手票與 C1／C2／C3 `e91`、`e96`、`e101`、`e102`、`e103`（用 Release 的 `finh_full`、`final_s0`、`c1_fit.npz`）→ `.scratch/e117/final/test_inputs.npz`。
- 第二階段（驗證框架）`e117`、`e124`：需要 `val_preds.npz`（驗證題六組機率，上游已不在，只能引用保存版）。

In [ ]:
for s in ("scripts/experiments/e10/pipeline.py", "scripts/experiments/e103/e103_make.py", "scripts/experiments/e117/e117_make.py"):
    show_doc(s, 1200)

## 段 8：最後一步

見 `colab/reproduce_final.ipynb`（需要主辦方提供的中間檔；約 1 分鐘重現最終提交檔並比對 sha256）。若你自己重算了上游，把產出放到與 `ASSETS.md` 相同的路徑，再用同樣的指令：

```
python final/make_rank_final.py --out out/rank_v13n.csv
```
（GPU 上游不會位元重現，所以自己重算的結果 sha256 會與上傳檔不同；差異大小請看 `PIPELINE.md` 與 README 的「已知限制」。）